# 🔍 Interactive YOLO Inference & Post-processing

This notebook provides an interactive UI to upload raw images, run YOLOv12 instance segmentation, apply morphological post-processing, and visualize the hollow contours mimicking the CVAT annotation style.

In [ ]:
# Install required libraries if not present
!pip install -q ultralytics ipywidgets shapely opencv-python matplotlib

In [ ]:
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib import pyplot as plt

# Add root directory to python path to import our custom pipeline
sys.path.append(str(Path(os.path.abspath('')).parent.parent))

from src.inference import MaskPostProcessor, RockVisualizer
from ultralytics import YOLO

print("✅ Dependencies loaded successfully!")

## ⚙️ 1. Configurations & Model Loading
Ensure you specify the correct exact path to your trained model `best.pt`.

In [ ]:
# Update this path to your locally trained model weights
weights_path = "../../models/YOLO26m_Batch4_March_Dataset_SAHI/weights/best.pt"

try:
    print(f"Loading YOLO model from: {weights_path}")
    model = YOLO(weights_path)
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model! Check your path. Details: {e}")

## 🎛️ 2. Interactive Inference Setup
Run the cell below to display an interactive file uploader and slider controls for fine-tuning the segmentation algorithm.

In [ ]:
# Style and Layout
style = {'description_width': 'initial'}
layout = widgets.Layout(width='400px')

# UI Components
upload_btn = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image'
)

conf_slider = widgets.FloatSlider(
    value=0.25, min=0.05, max=0.95, step=0.05,
    description='Confidence:', style=style, layout=layout
)

iou_slider = widgets.FloatSlider(
    value=0.45, min=0.1, max=0.9, step=0.05,
    description='NMS IoU:', style=style, layout=layout
)

min_area_slider = widgets.IntSlider(
    value=300, min=0, max=5000, step=50,
    description='Min Area (px²):', style=style, layout=layout
)

epsilon_slider = widgets.FloatSlider(
    value=0.01, min=0.001, max=0.05, step=0.001, readout_format='.3f',
    description='Epsilon Ratio:', style=style, layout=layout
)

run_button = widgets.Button(
    description='Run Inference',
    button_style='success',
    icon='play'
)

output_plot = widgets.Output()

def run_interactive_inference(b):
    with output_plot:
        clear_output(wait=True)
        
        if not upload_btn.value:
            print("⚠️ Please upload an image first.")
            return
            
        print("⏳ Processing...")
        
        # 1. Read Uploaded Image
        # Handle ipywidgets file upload format (dict in newer versions, tuple in older)
        uploaded_file = list(upload_btn.value.values())[0] if isinstance(upload_btn.value, dict) else upload_btn.value[0]
        content = uploaded_file['content']
        
        # Decode byte array to cv2 image
        nparr = np.frombuffer(content, np.uint8)
        original_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        # 2. Run YOLO Inference
        results = model.predict(
            source=original_bgr, 
            conf=conf_slider.value, 
            iou=iou_slider.value, 
            verbose=False
        )[0]
        
        # 3. Post-Process the Masks
        post_processor = MaskPostProcessor(
            min_area=min_area_slider.value,
            epsilon_ratio=epsilon_slider.value,
            simplify_tol=2.0
        )
        
        polygons_by_class = {}
        
        if results.masks is not None:
            mask_data = results.masks.data.cpu().numpy()
            class_ids = results.boxes.cls.cpu().numpy()
            
            for mask, cls_id in zip(mask_data, class_ids):
                # Ensure mask is scaled back to original image shape
                if mask.shape != original_bgr.shape[:2]:
                    mask = cv2.resize(mask, (original_bgr.shape[1], original_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
                
                clean_polys = post_processor.process(mask)
                class_id_int = int(cls_id)
                
                if class_id_int not in polygons_by_class:
                    polygons_by_class[class_id_int] = []
                polygons_by_class[class_id_int].extend(clean_polys)
                
        # 4. Render Visualizations 
        visualizer = RockVisualizer(thickness=8)
        
        hollow_drawn = visualizer.draw_hollow(original_bgr, polygons_by_class)
        mask_only = visualizer.draw_mask_only(original_bgr.shape, polygons_by_class)
        
        # 5. Display
        visualizer.plot_comparison(original_bgr, hollow_drawn, mask_only, title_suffix="")
        plt.show()
        print("✅ Processing complete!")

# Attach click event
run_button.on_click(run_interactive_inference)

# Display UI Structure
ui = widgets.VBox([
    widgets.HTML("<b>Upload Sample Image</b>"),
    upload_btn,
    widgets.HTML("<hr><b>Model Settings</b>"),
    conf_slider,
    iou_slider,
    widgets.HTML("<hr><b>Post-processing Settings (Smoothing & De-noising)</b>"),
    min_area_slider,
    epsilon_slider,
    widgets.HTML("<br>"),
    run_button,
    output_plot
])

display(ui)